# LifeMate — Public Test Submission

Runs the complete public-test split with Gipformer ASR and writes `/kaggle/working/submission.jsonl`. Attach the competition data to this notebook before running it. The default LLM is Qwen3.5-2B for T4 memory limits.

In [ ]:
import os

# Set the allocator option before importing torch.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CUDA allocator:", os.environ["PYTORCH_CUDA_ALLOC_CONF"])

## 1. Check the Kaggle accelerator

The runner uses one worker per GPU when two are available, one GPU when only one is available, and CPU otherwise.

In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi is unavailable; checking the device count in the next section.")

## 2. Load the repository

Enable Kaggle Internet for the initial repository and model downloads. Inference itself uses the downloaded local model files.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if Path("main.py").is_file():
    REPO_DIR = Path.cwd().resolve()
elif Path("/kaggle/working/LifeMate/main.py").is_file():
    REPO_DIR = Path("/kaggle/working/LifeMate").resolve()
else:
    subprocess.run(["git", "clone", "https://github.com/NT-Loi/LifeMate.git", "/kaggle/working/LifeMate"], check=True)
    REPO_DIR = Path("/kaggle/working/LifeMate").resolve()

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repository:", REPO_DIR)

## 3. Install the existing model requirements

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "models/llm/requirements-qwen.txt",
    "-r", "models/asr/requirements-gipformer.txt",
], check=True)

## 4. Check data, select the model, and download pinned assets

Change `MODEL_ID` to `Qwen/Qwen3.5-4B` if desired and the available GPU memory permits it. This cell downloads the exact revisions used by the code.

In [ ]:
import torch
from huggingface_hub import snapshot_download
from config import MODE, DATA_ROOT
from models.llm.qwen import MODEL_REVISIONS
from models.asr.gipformer import DEFAULT_VERSION as GIPFORMER_VERSION
from models.asr.gipformer import MODEL_REVISIONS as GIPFORMER_REVISIONS, REPO_IDS as GIPFORMER_REPO_IDS

MODEL_ID = "Qwen/Qwen3.5-2B"
MAX_NEW_TOKENS = 512
if MODEL_ID not in MODEL_REVISIONS:
    raise ValueError(f"No pinned revision configured for {MODEL_ID}")

public_cfg = MODE["public_test"]
required_paths = [Path(public_cfg["data_path"]), Path(public_cfg["catalog_path"]), Path(public_cfg["tools_dir"])]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Public-test files are missing. Attach the competition data.\n" + ", ".join(missing))

gpu_count = torch.cuda.device_count()
GPU_IDS = list(range(min(gpu_count, 2)))
print(f"Data root: {DATA_ROOT}")
with Path(public_cfg["data_path"]).open(encoding="utf-8") as source:
    public_test_rows = sum(1 for line in source if line.strip())
print(f"Public-test rows: {public_test_rows}")
print(f"Visible GPUs: {gpu_count}; runner GPUs: {GPU_IDS or 'CPU'}")

snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISIONS[MODEL_ID])
snapshot_download(
    repo_id=GIPFORMER_REPO_IDS[GIPFORMER_VERSION],
    revision=GIPFORMER_REVISIONS[GIPFORMER_VERSION],
)
print("Pinned Qwen and Gipformer assets are cached.")

## 5. Run the full public-test split

There is no sample limit. The script keeps timestamped predictions, recognized transcripts, conversations, and run/worker logs under `eval_results/`. It only replaces the root submission after every row succeeds.

In [ ]:
import shutil
import subprocess
from datetime import datetime

OUTPUT_DIR = REPO_DIR / "eval_results" / f"public_test_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
command = [
    sys.executable, "-u", "main.py",
    "--model-id", MODEL_ID,
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--output-dir", str(OUTPUT_DIR),
]
if GPU_IDS:
    command.extend(["--gpus", ",".join(map(str, GPU_IDS))])
print("Command:", " ".join(command))
completed = subprocess.run(command, cwd=REPO_DIR, check=False)

def show_log_tail(log_path, max_lines=60):
    print(f"\n--- {log_path} ---\n")
    if not log_path.is_file():
        print("Log file was not created.")
        return
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(lines[-max_lines:]) or "(empty log)")

for log_path in [OUTPUT_DIR / "run.log", *sorted(OUTPUT_DIR.glob("worker_*.log"))]:
    show_log_tail(log_path)
if completed.returncode != 0:
    raise subprocess.CalledProcessError(completed.returncode, command)

repo_submission = REPO_DIR / "submission.jsonl"
KAGGLE_SUBMISSION = Path("/kaggle/working/submission.jsonl")
shutil.copy2(repo_submission, KAGGLE_SUBMISSION)
print("Submission ready:", KAGGLE_SUBMISSION)

## 6. Verify the submission and locate logs

In [ ]:
import json
from IPython.display import FileLink

with KAGGLE_SUBMISSION.open(encoding="utf-8") as source:
    submission_rows = [json.loads(line) for line in source if line.strip()]
print(f"Submission rows: {len(submission_rows)}")

print("Run artifacts:", OUTPUT_DIR)
print("Conversations:", OUTPUT_DIR / "conversations.txt")
print("Run log:", OUTPUT_DIR / "run.log")
print("Worker logs:", *sorted(OUTPUT_DIR.glob("worker_*.log")), sep="\n  ")
display(FileLink(str(KAGGLE_SUBMISSION)))